# 1. RBAC and Custom Roles

AZ-500 expects you to **implement** role-based access control (RBAC) — not just know what it is. You create custom roles, assign them at the right scope, and understand the permission model.

## Before you run this notebook

1. Run `uv sync` in the lab folder.
2. In VS Code, click the **kernel picker** at the top-right of this notebook and choose the interpreter from `.venv` (the one created by `uv`).
3. If the kernel isn't listed, reload the window (`Cmd+Shift+P` then *Reload Window*).

No Docker is needed for this notebook — everything runs as plain Python.

## RBAC essentials

Azure RBAC has four elements:

| Element | What it is | Example |
|---------|-----------|----------|
| **Security principal** | Who gets access | User, group, service principal, managed identity |
| **Role definition** | What they can do | `Actions`, `NotActions`, `DataActions`, `NotDataActions` |
| **Scope** | Where it applies | Management group -> Subscription -> Resource group -> Resource |
| **Assignment** | Binding principal + role + scope | *"Alice is Contributor on rg-prod"* |


In [ ]:
import json

# Azure RBAC scope hierarchy. Roles assigned higher are INHERITED downward.
SCOPE_HIERARCHY = {
    'Management Group: Contoso': {
        'Subscription: Production': {
            'Resource Group: rg-web':  ['App Service: web-app', 'SQL DB: web-db', 'Key Vault: web-kv'],
            'Resource Group: rg-data': ['Storage: datalake', 'Synapse: analytics'],
        },
        'Subscription: Development': {
            'Resource Group: rg-dev':  ['App Service: dev-app', 'SQL DB: dev-db'],
        },
    },
}

def show_scope(scope, indent=0):
    prefix = '  ' * indent
    if isinstance(scope, dict):
        for k, v in scope.items():
            print(f'{prefix}[FOLDER] {k}')
            show_scope(v, indent + 1)
    elif isinstance(scope, list):
        for item in scope:
            print(f'{prefix}  - {item}')

print('=== Azure RBAC scope hierarchy ===')
print('Roles at a higher scope are INHERITED by all children.\n')
show_scope(SCOPE_HIERARCHY)
print('\nIf Alice is "Contributor" on the Production subscription,')
print('she has Contributor on EVERY resource in rg-web and rg-data.')


## Built-in roles

Azure has 300+ built-in roles. Key ones for the exam:

| Role | Permissions | Typical use |
|------|-----------|----------------|
| **Owner** | Everything + assign roles | Subscription admins |
| **Contributor** | Everything *except* assign roles | DevOps teams |
| **Reader** | Read-only | Auditors |
| **User Access Administrator** | Manage role assignments only | Delegating access management |
| **Key Vault Administrator** | Full Key Vault management | Ops teams |
| **Key Vault Secrets User** | Read secrets only | Applications |
| **Storage Blob Data Contributor** | Read/write blob **data** | Apps accessing storage |
| **Network Contributor** | Manage networking | Network team |

### Control plane vs data plane

This distinction is critical for AZ-500:

- **Control plane** (`Actions` / `NotActions`): manage the resource itself (create, delete, configure).
- **Data plane** (`DataActions` / `NotDataActions`): access the data *inside* the resource (read blobs, get secrets).

**Contributor has full control-plane access but ZERO data-plane access** to Key Vault or Storage. That's why you still need `Key Vault Secrets User` or `Storage Blob Data Contributor` on top.


In [ ]:
# A tiny simulation of Azure's permission engine.
ROLES = {
    'Contributor': {
        'Actions': ['*'],
        'NotActions': [
            'Microsoft.Authorization/roleAssignments/*',
            'Microsoft.Authorization/roleDefinitions/*',
        ],
        'DataActions': [],
        'NotDataActions': [],
    },
    'Storage Blob Data Contributor': {
        'Actions': [
            'Microsoft.Storage/storageAccounts/blobServices/containers/read',
            'Microsoft.Storage/storageAccounts/blobServices/containers/write',
            'Microsoft.Storage/storageAccounts/blobServices/containers/delete',
        ],
        'NotActions': [],
        'DataActions': ['Microsoft.Storage/storageAccounts/blobServices/containers/blobs/*'],
        'NotDataActions': [],
    },
    'Key Vault Secrets User': {
        'Actions': [],
        'NotActions': [],
        'DataActions': [
            'Microsoft.KeyVault/vaults/secrets/getSecret/action',
            'Microsoft.KeyVault/vaults/secrets/readMetadata/action',
        ],
        'NotDataActions': [],
    },
}

def _matches(pattern: str, op: str) -> bool:
    # '*' matches anything. Trailing '/*' matches any child path.
    if pattern == '*':
        return True
    if pattern.endswith('/*'):
        return op.startswith(pattern[:-1])
    return pattern == op

def check_permission(role_name: str, operation: str, plane: str) -> str:
    role = ROLES[role_name]
    if plane == 'control':
        actions, not_actions = role['Actions'], role['NotActions']
    else:
        actions, not_actions = role['DataActions'], role['NotDataActions']
    if any(_matches(p, operation) for p in not_actions):
        return 'DENIED (NotActions)'
    if any(_matches(p, operation) for p in actions):
        return 'ALLOWED'
    return 'NOT PERMITTED (no matching action)'

print('=== Permission checks ===\n')
checks = [
    ('Contributor', 'Microsoft.Compute/virtualMachines/delete', 'control', 'Delete a VM'),
    ('Contributor', 'Microsoft.Authorization/roleAssignments/write', 'control', 'Assign a role'),
    ('Contributor', 'Microsoft.KeyVault/vaults/secrets/getSecret/action', 'data', 'Read a Key Vault secret'),
    ('Key Vault Secrets User', 'Microsoft.KeyVault/vaults/secrets/getSecret/action', 'data', 'Read a Key Vault secret'),
    ('Storage Blob Data Contributor',
     'Microsoft.Storage/storageAccounts/blobServices/containers/blobs/read', 'data', 'Read a blob'),
]
for role, op, plane, desc in checks:
    result = check_permission(role, op, plane)
    print(f'{result:<28}  {role:<35}  ->  {desc}')


## Bad to Best: least privilege in action

Real-world scenario: *"Alice is on the VM support team. She needs to restart VMs in the production resource group when there is an incident."*

Let's see three approaches, from worst to best, and check each one against the principle of **least privilege**.


In [ ]:
# Each row is (label, role, scope). Lower privilege is better.
SCENARIOS = [
    ('BAD:    Owner on subscription',            'Owner',       '/subscriptions/SUB'),
    ('OKAY:   Contributor on resource group',    'Contributor', '/subscriptions/SUB/resourceGroups/rg-prod'),
    ('BEST:   Custom "VM Restart Operator" on resource group',
                                                 'VM Restart',  '/subscriptions/SUB/resourceGroups/rg-prod'),
]

# What can Alice do with each role?
#                       read   restart  delete  create_vm  assign_roles  read_secrets
CAPABILITIES = {
    'Owner':          ( True,  True,    True,   True,      True,         True ),
    'Contributor':    ( True,  True,    True,   True,      False,        False),
    'VM Restart':     ( True,  True,    False,  False,     False,        False),
}
LABELS = ['View VM', 'Restart', 'Delete', 'Create', 'Assign roles', 'Read KV secrets']

header = f'{"Approach":<62} ' + '  '.join(f'{l:<15}' for l in LABELS)
print(header)
print('-' * len(header))
for label, role, _scope in SCENARIOS:
    caps = CAPABILITIES[role]
    row = '  '.join(f'{("yes" if c else "no"):<15}' for c in caps)
    print(f'{label:<62} {row}')

print('''
Takeaway:
  * Owner gives Alice the power to DELETE production VMs and even reassign roles to herself.
  * Contributor still lets her delete or recreate VMs - one typo could cause an outage.
  * A narrow custom role scoped to rg-prod gives her exactly what she needs and nothing more.
    If her account is ever compromised, the blast radius is tiny.
''')


## Custom role JSON definition

When no built-in role fits, define a custom role. This is the JSON Azure expects.


In [ ]:
custom_role = {
    'Name': 'VM Restart Operator',
    'Description': 'View and restart VMs. Cannot create or delete them.',
    'Actions': [
        'Microsoft.Compute/virtualMachines/read',
        'Microsoft.Compute/virtualMachines/restart/action',
        'Microsoft.Compute/virtualMachines/start/action',
        'Microsoft.Compute/virtualMachines/powerOff/action',
        'Microsoft.Resources/subscriptions/resourceGroups/read',
    ],
    'NotActions': [],
    'DataActions': [],
    'NotDataActions': [],
    'AssignableScopes': ['/subscriptions/00000000-0000-0000-0000-000000000000'],
}

print('Custom role JSON:')
print(json.dumps(custom_role, indent=2))

print('\n--- Azure CLI: create the role ---')
print('az role definition create --role-definition @vm-restart-operator.json')

print('\n--- Azure CLI: assign the role ---')
print('az role assignment create \\')
print('  --assignee alice@contoso.com \\')
print('  --role "VM Restart Operator" \\')
print('  --scope /subscriptions/.../resourceGroups/rg-prod')

# Quick sanity check: what does this role allow?
print('\n--- What Alice can and cannot do with this role ---')
tests = [
    ('Microsoft.Compute/virtualMachines/read',            'View VMs'),
    ('Microsoft.Compute/virtualMachines/restart/action',  'Restart VMs'),
    ('Microsoft.Compute/virtualMachines/delete',          'Delete VMs'),
    ('Microsoft.Compute/virtualMachines/write',           'Create/update VMs'),
]
for op, desc in tests:
    allowed = op in custom_role['Actions']
    print(f'  {"yes" if allowed else "no":<4} {desc}')


## Access Reviews - keep permissions fresh

Permissions accumulate over time ("privilege creep"). **Access Reviews** (in Entra ID Governance) periodically ask an owner: *"Does Alice still need Contributor on rg-prod?"*

Typical exam-worthy setup:

| Setting | Recommended value |
|---------|-------------------|
| **Scope** | Privileged roles, guest users, high-sensitivity groups |
| **Reviewers** | Group owner, resource owner, or the user themselves ("self-review") |
| **Frequency** | Quarterly for standing access, monthly for privileged |
| **Auto-apply results** | Yes - remove access if not approved |
| **If reviewer doesn't respond** | Remove access (safer default) |

```bash
# Create an access review for the "Contributor" role on a subscription.
az rest --method POST \
  --uri 'https://graph.microsoft.com/v1.0/identityGovernance/accessReviews/definitions' \
  --body @access-review.json
```

## Deny assignments

Azure also supports **deny assignments** that *block* access even if a role grants it.
They are created by Azure Blueprints or managed apps - you cannot create them directly.
**Deny always wins over allow.**

---
## Summary

| Concept | What to remember |
|---------|-------------------|
| **Scope hierarchy** | MG -> Sub -> RG -> Resource. Roles inherit downward. |
| **Control vs data plane** | `Actions` manages the resource. `DataActions` accesses the data inside. |
| **Contributor gap** | Full control-plane, zero data-plane. Add `Key Vault Secrets User` etc. |
| **Least privilege** | Prefer a narrow custom role at the smallest scope over a broad built-in role. |
| **Access Reviews** | Fight privilege creep. Quarterly for standing access, monthly for privileged. |
| **Deny assignments** | Always win. Created by Blueprints, not users. |

**Next**: [Notebook 2 - PIM and Conditional Access](02_pim_and_conditional_access.ipynb)
